# COMP0178 — Lesson 4: Query Processing and Optimization

- Source: [Lecture 4: Query Processing](../Lectures/Lecture_4_Query_Processing.pdf), all **41 PDF pages**. References below use PDF page order; some slide footer numbers differ, so follow the PDF-page references.
- Goal: understand how a DBMS translates SQL into relational algebra (RA), checks it, and chooses an equivalent query plan with lower estimated cost.
- The lecture's cost examples count tuple reads/writes under stated simplified assumptions. They are teaching models, not universal DBMS timing predictions.
- Exam links refer to the exact supplied papers; the longer end-to-end exam solutions are in [Lesson 3b: Relational Algebra](lesson3b.ipynb).


## 1. The big picture — PDF pages 2–3

SQL is **declarative**: you say what rows and columns you want, not the detailed order in which the DBMS must fetch and combine them. The DBMS can therefore choose a plan.

Query processing has two broad jobs:

1. **Query decomposition:** check the SQL and translate it into an equivalent relational-algebra expression/tree.
2. **Query optimization:** find an equivalent expression/tree expected to do less work, often by reducing disk I/O.

For example, the business question “Which managers work at a London branch?” can be written as SQL. The DBMS may evaluate the same meaning by first combining all staff and branches, or by first filtering to managers and London branches, then joining. Those plans return the same answer but can have very different costs.

**Original lecture visual — PDF page 3**

[![Query processing separates decomposition from optimization.](../assets/slide-images/Lecture_4_Query_Processing-page-03.png)](../assets/slide-images/Lecture_4_Query_Processing-page-03.png)

Source: [Lecture 4, PDF page 3](../Lectures/Lecture_4_Query_Processing.pdf#page=3).


## 2. Query decomposition: four stages — PDF pages 5–17

The lecture divides decomposition into **parsing, normalization, semantic analysis, and simplification**. These are logical stages; a real DBMS may organize its internals differently.

### 2.1 Parsing — pages 6–10

Parsing checks that the query is written in a recognizable form and builds a representation of its structure.

- **Lexical analysis:** split the text into tokens such as `SELECT`, `staffNo`, `FROM`, `Staff`, `WHERE`, `>`, and `20000`. Each token must be recognizable.
- **Syntactic analysis:** check that tokens follow SQL grammar—for example, a `SELECT` list followed by `FROM` and a valid condition.
- **Generic semantic checks:** check names and basic type compatibility. Does the table or column exist? Is a numerical comparison being applied to a number?

A parser can turn valid syntax into an **abstract syntax tree (AST)**, which records the query's structure. It is not yet a physical execution plan.

Example from the lecture:

```sql
SELECT staffNumber
FROM Staff
WHERE position > 10;
```

If the schema has `staffNo`, not `staffNumber`, the name is invalid. If `position` is text, comparing it numerically to `10` is type-inappropriate.

Once checks pass, the query can be represented as a relational-algebra tree. In an RA tree, **leaves are input relations**, internal nodes are operations, and the **root is the final operation**.

**Original lecture visuals — PDF pages 7–10**

[![Tokens and grammar lead to an abstract syntax tree.](../assets/slide-images/Lecture_4_Query_Processing-page-07.png)](../assets/slide-images/Lecture_4_Query_Processing-page-07.png)

[![An RA tree has relations at leaves and operations above them.](../assets/slide-images/Lecture_4_Query_Processing-page-10.png)](../assets/slide-images/Lecture_4_Query_Processing-page-10.png)


### 2.2 Normalization — page 11

Normalization rewrites Boolean conditions into a standard shape. It makes later reasoning about conditions systematic.

- **Conjunctive normal form (CNF):** an AND of clauses, where each clause is an OR. Example: `(p OR q) AND r AND (s OR t)`.
- **Disjunctive normal form (DNF):** an OR of clauses, where each clause is an AND. Example: `(p AND q) OR r OR (s AND t)`.

The names tell you the outer connective: CNF is a conjunction (AND) of disjunctions; DNF is a disjunction (OR) of conjunctions.


### 2.3 Semantic analysis — pages 12–16

Semantic analysis looks for queries that are syntactically valid but logically defective.

**Disconnected relations.** Make a relation-connection graph. Include each relation in the query. Connect relations joined by a condition, and connect a relation to the result when its attributes are requested. If some mentioned relation is disconnected from the result, it may be unnecessary or indicate a missing join condition. The lecture calls this an incorrectly formulated query.

In the lecture's example (PDF pages 14–16), Client and Viewing are connected by `clientNo`, and Viewing and PropertyForRent by `propertyNo`. If the query mentions a relation without linking it to the rest, the graph exposes that issue.

**Contradictory conditions.** A query can also be impossible to satisfy, such as requiring `maxRent >= 500` and `maxRent < 200` at once. The lecture introduces a normalized attribute-connection graph: attributes and a constant node are connected by weighted directed edges representing inequalities. If a cycle has a negative total weight, the constraints are inconsistent, so no tuple can satisfy them. This is a graph-based way to detect contradictions.

**Original lecture visuals — PDF pages 14–16**

[![A relation connection graph checks that all query relations contribute to a connected result.](../assets/slide-images/Lecture_4_Query_Processing-page-14.png)](../assets/slide-images/Lecture_4_Query_Processing-page-14.png)

[![Attribute constraints can expose a contradictory cycle.](../assets/slide-images/Lecture_4_Query_Processing-page-16.png)](../assets/slide-images/Lecture_4_Query_Processing-page-16.png)


### 2.4 Simplification and access checks — page 17

After validity checks, the system can check whether the user has permission to access the referenced objects, then simplify the condition using Boolean laws and known integrity constraints. For example, if a constraint says an attribute can never have value `X`, a condition requiring that value is always false.

Do not confuse these stages: parsing asks whether the query is formed properly; semantic analysis asks whether its names, types, and logic make sense; simplification rewrites a valid query more simply.


## 3. Why optimize? — PDF pages 19–23

A single SQL query can have many equivalent relational-algebra plans. A plan that creates a huge intermediate relation may do much more I/O than one that filters early.

Lecture example: “Find managers who work at a London branch.”

```sql
SELECT *
FROM Staff s, Branch b
WHERE s.branchNo = b.branchNo
  AND s.position = 'Manager'
  AND b.city = 'London';
```

Let `S = Staff`, `B = Branch`, `p = (position='Manager')`, `q = (city='London')`, and `j = (S.branchNo=B.branchNo)`. Three equivalent RA forms are:

```text
1. σ[p AND q AND j](S × B)
2. σ[p AND q](S ⋈[j] B)
3. σ[p](S) ⋈[j] σ[q](B)
```

Plan 1 forms the Cartesian product before filtering. Plan 2 uses a join instead of forming all pairs. Plan 3 filters each table first, then joins only the smaller sets.

Under the lecture's assumptions—1,000 staff, 50 branches, 50 managers, 5 London branches, no indexes, and tuple-at-a-time costs—the slide estimates:

| Plan | Estimated disk accesses | Why |
|---|---:|---|
| 1 | 101,050 | Read the inputs, write and scan the full product, then select. |
| 2 | 3,050 | Read the inputs and process the much smaller join result. |
| 3 | 1,160 | Filter each base relation first and join the small filtered sets. |

These figures belong to the lecture's simplified cost model. Real costs depend on indexes, memory, join algorithms, data sizes, selectivity, and whether intermediates are materialized. State the model before calculating.

**Original lecture visuals — PDF pages 20–22**

[![Three equivalent plans for managers at London branches.](../assets/slide-images/Lecture_4_Query_Processing-page-20.png)](../assets/slide-images/Lecture_4_Query_Processing-page-20.png)

[![The plans have very different estimated disk costs.](../assets/slide-images/Lecture_4_Query_Processing-page-21.png)](../assets/slide-images/Lecture_4_Query_Processing-page-21.png)

### Dynamic and static optimization — page 23

- **Dynamic optimization** chooses a plan at run time, using current statistics. It may produce a better-informed plan but adds planning time.
- **Static optimization** chooses a plan in advance. It avoids that run-time work but may use stale statistics.
- **Hybrid approaches** can re-optimize after significant changes or optimize once per session and cache the plan.


## 4. RA equivalences that enable optimization — PDF pages 25–33

The optimizer uses algebraic laws to change a tree without changing the query's meaning. Here, `σ` means selection (filter rows), `π` means projection (keep columns), `×` is Cartesian product, and `⋈` is join.

### 4.1 Basic identities — page 26

```text
R ∩ S = R − (R − S)
R ⋈[p] S = σ[p](R × S)
```

A join is conceptually a Cartesian product followed by a matching condition. A DBMS need not physically build the full product when it executes a join.

### 4.2 Split and reorder selections — page 27

```text
σ[p AND q](R) = σ[p](σ[q](R)) = σ[q](σ[p](R))
```

Filtering by `p` and then by `q` gives the same rows as filtering by both at once. This lets the optimizer push each filter down to the relation it uses.

### 4.3 Combine or split projections — page 28

For valid projections, nested projections can collapse to the final set of columns:

```text
π[A](π[B](R)) = π[A](R), when A ⊆ B
```

More usefully, an optimizer can project early to keep only columns needed for the final result **and** for later join/filter conditions. Keep join keys until the join has happened.

### 4.4 Move selection across projection — page 29

```text
π[A](σ[p](R)) = σ[p](π[A](R)), when every attribute used by p is in A
```

The condition can only be evaluated after a projection if that projection kept the columns the condition needs. If `p` mentions `salary` but the projection drops `salary`, the right side is invalid.

### 4.5 Commutative operations — page 30

Products, joins, union, and intersection are commutative: swapping operands does not change the mathematical result (though the output column order can change). Set difference is **not** commutative: generally `R − S ≠ S − R`.

### 4.6 Push selections across operations — page 31

For compatible relation schemas, a selection can be distributed over union, intersection, or difference. For a product/join, split a conjunction when each part refers only to its own input:

```text
σ[p AND q](R × S) = σ[p](R) × σ[q](S)
```

where `p` uses only attributes of `R` and `q` only attributes of `S`. The same pushdown applies to a join, provided join predicates and attributes are preserved.

### 4.7 Push projections across operations — page 32

Projection can be pushed toward the leaves to reduce row width, but must keep all columns needed later. Across a join, preserve:

- output columns required by the final answer;
- join-key columns used to match the relations;
- any columns needed by filters above the join.

Dropping a join key too early makes the join impossible. Dropping a filter column too early makes the filter impossible.

### 4.8 Associativity — page 33

Union, intersection, Cartesian product, and natural join are associative. Difference is not. Associativity lets an optimizer choose which join to do first:

```text
(R ⋈ S) ⋈ T  ≡  R ⋈ (S ⋈ T)
```

For conditional joins, carry the correct predicates with the relevant relations when reordering.

**Original lecture visuals — PDF pages 26–33**

[![Selection and projection transformations.](../assets/slide-images/Lecture_4_Query_Processing-page-26.png)](../assets/slide-images/Lecture_4_Query_Processing-page-26.png)

[![Commutative and distributive rules for binary operations.](../assets/slide-images/Lecture_4_Query_Processing-page-30.png)](../assets/slide-images/Lecture_4_Query_Processing-page-30.png)

[![Associativity allows the order of joins to change.](../assets/slide-images/Lecture_4_Query_Processing-page-32.png)](../assets/slide-images/Lecture_4_Query_Processing-page-32.png)


## 5. Optimize a three-table query — PDF pages 35–40

The lecture asks for prospective flat renters whose budget and preferred type match a property owned by `CO93`. It involves Client `C`, Viewing `V`, and PropertyForRent `P`.

```sql
SELECT p.propertyNo, p.street
FROM Client c, Viewing v, PropertyForRent p
WHERE c.prefType = 'Flat'
  AND c.clientNo = v.clientNo
  AND v.propertyNo = p.propertyNo
  AND c.maxRent >= p.rent
  AND c.prefType = p.type
  AND p.ownerNo = 'CO93';
```

### Step 1: Translate SQL to algebra

```text
π[P.propertyNo, P.street](
  σ[C.prefType='Flat' AND C.clientNo=V.clientNo
    AND V.propertyNo=P.propertyNo AND C.maxRent≥P.rent
    AND C.prefType=P.type AND P.ownerNo='CO93'](C × V × P))
```

The outer `π` returns only the requested property number and street. The `σ` selects rows satisfying the WHERE conditions. The product provides combinations of rows to which the predicates apply.

### Step 2: Split predicates and push them down

Predicates using only one relation can be applied before joining:

```text
σ[C.prefType='Flat'](C)
σ[P.ownerNo='CO93'](P)
```

Join predicates and cross-table comparisons must wait until their required columns are available. The join keys and columns for comparisons (`clientNo`, `propertyNo`, `maxRent`, `rent`, `prefType`, `type`) must not be projected away prematurely.

### Step 3: Replace product-plus-equality with joins

`C × V` followed by `C.clientNo=V.clientNo` is an equijoin. Likewise, connect V to P using propertyNo. Then use join associativity to choose a useful order.

If only a small number of properties are owned by CO93, filter P early and join that small set first. The best first join depends on cardinalities/selectivities; “most selective first” is a useful heuristic, not a guaranteed rule for all cost models.

### Step 4: Project columns early, but safely

After early filters, each input can drop columns that will never be used. Keep needed join keys and comparison attributes until their operations are complete. At the end, keep only `P.propertyNo` and `P.street`.

**Original lecture visuals — PDF pages 36–40**

[![Push selections toward the relation leaves.](../assets/slide-images/Lecture_4_Query_Processing-page-36.png)](../assets/slide-images/Lecture_4_Query_Processing-page-36.png)

[![Convert product plus a matching condition into an equijoin.](../assets/slide-images/Lecture_4_Query_Processing-page-37.png)](../assets/slide-images/Lecture_4_Query_Processing-page-37.png)

[![Choose a join order using selectivity.](../assets/slide-images/Lecture_4_Query_Processing-page-38.png)](../assets/slide-images/Lecture_4_Query_Processing-page-38.png)

[![Push safe projections inward.](../assets/slide-images/Lecture_4_Query_Processing-page-39.png)](../assets/slide-images/Lecture_4_Query_Processing-page-39.png)

### Lecture heuristics — page 40

1. Apply selections and projections as early as safely possible; they reduce intermediate relation size.
2. Push operations inward using equivalence rules.
3. Replace a Cartesian product followed by a matching selection with a join.
4. Apply the most restrictive filters early when that reduces later work.
5. Reorder associative operations to choose a useful join order.
6. Compute a common subexpression once and reuse it when storing the result is worthwhile.

A heuristic is a practical guide; a cost-based optimizer still needs estimates and may find a different plan is cheaper.


## 6. SQL bag semantics versus classical RA sets — supplementary

Classical relational algebra treats relations as **sets**, so duplicate tuples disappear. SQL normally uses **bag (multiset) semantics**, so a query may return duplicates unless `DISTINCT` is requested. This matters when comparing an SQL query with RA projection.

- If you need SQL behavior with duplicates, do not silently assume set projection.
- If the intended result is unique identifiers, use `SELECT DISTINCT` or explicitly state the set semantics in RA.
- `COUNT`, `SUM`, `AVG`, `MIN`, and `MAX` require extended RA notation; basic σ, π, and joins alone do not express aggregation.


## 7. Runnable SQL practice — supplementary

The sample SQLite database is a small local practice schema. It is not the exact lecture dataset. SQLite is used so no MySQL server is needed for these exercises.


In [ ]:
from pathlib import Path
import sys
practice_folder = next((p for p in [Path.cwd(), Path.cwd().parent,
    Path.cwd() / "COMP0178", Path("/Users/peidi/Desktop/COMP0178")]
    if (p / "sql_practice.py").exists()), None)
if practice_folder is None:
    raise FileNotFoundError("Keep sql_practice.py in the COMP0178 folder.")
sys.path.insert(0, str(practice_folder))
from sql_practice import Practice
lab = Practice()
lab.register(get_ipython())

### Practice 1: selection then projection

Business question: which staff numbers and first names belong to staff earning more than 20,000? Translate the query to RA. Think about whether the final result should contain duplicates.


In [ ]:
%%sql
SELECT DISTINCT staffNo, fName
FROM Staff
WHERE salary > 20000
ORDER BY staffNo;

In [ ]:
lab.check([("S001", "Ada"), ("S003", "Cara")], ordered=True)

**RA answer:** `π[staffNo,fName](σ[salary>20000](Staff))`. The filter uses salary, so selection happens before projection, then only requested columns remain.


### Practice 2: optimize without changing the answer

The SQL is already written with a join. Find managers at London branches. Write an equivalent expression that filters Staff and Branch before joining. Keep the branch key in both inputs so the join can still happen.


In [ ]:
%%sql
SELECT DISTINCT s.staffNo, s.fName
FROM Staff s JOIN Branch b ON s.branchNo = b.branchNo
WHERE s.position = 'Manager' AND b.city = 'London'
ORDER BY s.staffNo;

In [ ]:
lab.check([("S001", "Ada")], ordered=True)

**One optimized RA form:**

```text
π[S.staffNo,S.fName](
  σ[S.position='Manager'](Staff)
  ⋈[S.branchNo=B.branchNo]
  σ[B.city='London'](Branch))
```

The filter on each table is pushed to that table before the join. Keep `branchNo` through the join; the final projection can remove it afterward.


## 8. Costing a query plan — supplementary method

For exam-style estimates, write the assumptions and the plan before adding numbers:

1. Record base relation sizes and which tables the query reads.
2. Estimate each intermediate result using the question's selectivity information. If missing, state a reasonable assumption; do not present it as a fact.
3. Under a materialization model, count reads of each input and any intermediate, plus writes of each nonfinal intermediate if the question asks total accesses.
4. Distinguish **reads** from **reads + writes**. Follow the paper's exact convention, including whether final output is written and whether memory avoids rescanning inputs.
5. Compare equivalent plans under the same assumptions.

There is no cost number independent of a model. Selectivity, indexing, memory, tuple widths, join algorithms, and materialization can change the answer.


## 9. Exam connection — query processing

The supplied past papers repeatedly assess the chain **business question → SQL → RA expression → two equivalent RA trees → estimated costs**:

- **2023 Q2(a–d):** formulate a query, express it in RA, draw equivalent trees, and estimate disk reads. See [the 2023 paper, Q2, PDF pages 2–3](../Exampapers/COMP0178A7PE.pdf#page=2) and the worked solution in [Lesson 3b, section 16](lesson3b.ipynb).
- **2024 Q2(a–c):** use all three surgery tables, write SQL/RA, draw two equivalent trees, and estimate costs. See [the 2024 paper, Q2, PDF pages 2–3](../Exampapers/COMP0178A7PF.pdf#page=2) and [Lesson 3b, section 17](lesson3b.ipynb). The paper labels Q2 as 30 marks even though its listed subparts total 20; the notes preserve that discrepancy.
- **2025 Q2(a–d):** construct a different business question from the required tables, then SQL, RA, trees, and costs. See [the 2025 paper, Q2, PDF pages 2–4](../Exampapers/COMP0178A7PG.pdf#page=2) and [Lesson 3b, section 18](lesson3b.ipynb).

For any such question, annotate tree nodes clearly. If you push a selection down, prove its predicate only uses that relation. If you push a projection down, show that all output, filter, and join attributes survive until they are no longer needed.


## 10. Common mistakes and revision answers

1. **What does decomposition do?** Checks and translates the SQL into a valid RA representation.
2. **What does optimization do?** Chooses an equivalent expression expected to cost less.
3. **What is a disconnected relation graph warning?** A table mentioned in the query is not connected by joins/used output to the result; it may be unnecessary or a join condition may be missing.
4. **When can a selection move below a join?** When the predicate uses only attributes of the relation being filtered; preserve the join predicate separately.
5. **When can projection move early?** When it keeps all eventual output attributes, filter attributes, and join keys needed later.
6. **Are `R−S` and `S−R` equivalent?** No, difference is not commutative.
7. **Why avoid a Cartesian product?** It can create every pair of tuples before filtering, producing a large intermediate.
8. **Does a relational-algebra tree determine physical execution exactly?** No. It describes logical operations; a DBMS may execute them with different physical algorithms.
9. **Does SQL projection always remove duplicates?** No. SQL keeps duplicates by default; classical RA projection is set-based.
10. **What must a cost answer state?** The cost model, cardinality/selectivity assumptions, intermediate materialization behavior, and whether it reports reads only or reads plus writes.


## 11. Full lecture map

| PDF pages | Topic |
|---|---|
| 1–3 | Learning objectives and query processing in an RDBMS |
| 4–10 | Decomposition, parsing, syntax trees, and RA trees |
| 11–17 | Normalization, semantic checks, contradictions, and simplification |
| 18–23 | Optimization, equivalent plans, costs, static/dynamic approaches |
| 24–33 | RA equivalence and transformation rules |
| 34–40 | Worked three-table optimization and heuristics |
| 41 | Summary |
